# Step 2c · Event library and outage labels

**Purpose**: produce the "county × event" outage label table for the modeling, and screen the candidate hurricanes.

Two choices about the data matter most:

1. **The denominator uses the 2022 customers per person**: in EAGLE-I's `coverage_history.csv`, the customer total of Texas is 18.56–19.62 million for 2018–2020 and 14.69–14.80 million for 2021–2022, a difference of about 30%, while the population did not change like that. Taking the figure of the nearest year would make the Texas denominator for Laura (2020) about one third too large. So the 2022 "statewide customers ÷ state population" is used for every event, multiplied by the county population of the event's own year
2. **Four cleaning measures**: removing spikes shorter than 1 hour, flagging counties without records and removing states without data, using the peak as the denominator when it exceeds the estimated customer total, and using `hours_rel` as the denominator-free indicator

Outage records already extracted to `eaglei_<event>.csv` in the output folder are read from there; a missing one is downloaded again. Result file names all start with `step2c_`.

**Structure**

| Part | Content | Needs Earth Engine | Approx. time |
| --- | --- | --- | --- |
| A | Event library: 11 candidate hurricanes | No | 1 minute |
| B | Read the outage records already extracted | No | 1 minute |
| C | Customers per county (the denominator) | No | 1 minute |
| D | Cleaning; build the 15-minute panel for each event | No | 2–4 minutes |
| E | "County × event" label table and three figures | No | 2 minutes |
| F | Sentinel-1 overpass check | Only for counties without a saved result | A few minutes |
| G | Event screening table and checklist | No | 1 minute |

**Requirements**

- An Earth Engine Cloud project, set as `PROJECT_ID` in cell 2, is used in part F only. Without it, the Sentinel-1 result for counties without a saved result stays empty; the other results are not affected

In [ ]:
# 1. Environment, output folder, chart style
import os, time
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, TMP = "/content/drive/MyDrive/us_flood_outage", "/content"
except ImportError:                     # in local Jupyter: use the current folder
    OUT = "us_flood_outage" if os.path.isdir("us_flood_outage") else "../us_flood_outage"    # next to the notebook, or one level up
    TMP = f"{OUT}/tmp"
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
print("Output folder:", OUT)

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

def spearman(a, b):
    # Spearman rank correlation: rank first, then take the ordinary correlation
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank())

# Chart style (shared by all figures)
C_POWER = "#2a78d6"
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## Parameters

These thresholds are the same for every event. After changing any of them, rerun from part D.

In [ ]:
# 2. Parameters
PROJECT_ID = "ee-yourname"       # Earth Engine Cloud project ID (used in part F only)
RERUN_S1   = False               # False = existing Sentinel-1 results are reused and only missing counties are computed; True = compute everything again (about 50 minutes)

STEP_H     = 0.25                # one time slot = 15 minutes
OUT_THRESH = 0.10                # "substantial outage": at least 10% of the county's customers are out
REL_THRESH = 0.10                # relative threshold: customers out reach 10% of the county's own peak
CALM_HOURS = 24                  # restored = stays below the threshold for this many hours
MIN_PEAK   = 100                 # a county is "affected" only with at least this many customers out at the peak (rules out noise in tiny counties)
PRE_DAYS   = 3                   # each event window starts this many days before landfall
PEAK_DAYS  = 7                   # the outage peak is searched within this many days after landfall
DEN_ERR    = 0.25                # sensitivity check: assume the customer totals are this much too high or too low

GAP_FILL_H     = 12              # collection gaps up to this long are filled by interpolation; longer ones stay empty
DROP_WINDOW_H  = 6               # time window (hours) used as the reference when detecting sharp drops
DROP_RATIO     = 0.5             # a value below this fraction of the surrounding median counts as a drop
SPIKE_WINDOW_H = 2               # time window (hours) used as the reference when detecting spikes; only bumps shorter than half of it count as spikes
SPIKE_RATIO    = 1.5             # a value above this multiple of the surrounding median counts as a spike
DROP_MIN_SHARE = 0.02            # a value is corrected only when the deviation reaches this share of the county's customers (everyday small fluctuations are left alone)

NO_RECORD_MAX = 0.20             # when more than this share of a state's counties have no record at all in an event, the state is treated as not covered at the time

S1_DAYS      = 6                 # a radar image counts as "timely" within this many days after landfall
S1_MIN_SHARE = 0.5               # at least this share of an event's affected counties must have a timely image

## A. Event library

The candidates are 11 hurricanes that made landfall on the U.S. Southeast and Gulf coasts in 2017–2024 and caused widespread outages. Landfall time, place and intensity all come from the official reports of the U.S. National Hurricane Center (NHC); the `report` column links to each report.

- `states`: which states to extract for the event (FIPS codes). The scope is limited to 8 states on the Southeast and Gulf coasts, and every one of them that the hurricane crossed is listed
- `days`: how many days after landfall to observe. Events with very long outages (Michael, Laura, Ida, Helene) get 30 days, the others 21
- `prior`: the main hazard described in the report (`flood`, `wind` or `mixed`). This is only a rough prior judgment; Step 3 replaces it with wind and flooding measured for each county

**One special case**: 13 days after Helene's landfall in 2024, Milton struck Florida as well. Helene's Florida data are therefore used only up to 9 October, and counties not yet restored by then are marked as "restoration not observed".

Which states are removed for lack of data is decided automatically from the data in part E, not edited by hand here.

In [ ]:
# 3. Event library (landfall times in UTC, from the official NHC reports)
EVENTS = pd.DataFrame([
    dict(key="harvey2017",   name="Harvey",   year=2017, landfall="2017-08-26 03:00", place="San Jose Island, TX",     cat="Cat 4", states=["48", "22"],                         days=21, prior="flood", atcf="AL092017"),
    dict(key="irma2017",     name="Irma",     year=2017, landfall="2017-09-10 13:00", place="Cudjoe Key, FL",          cat="Cat 4", states=["12", "13", "45", "01", "37"],       days=21, prior="wind",  atcf="AL112017"),
    dict(key="florence2018", name="Florence", year=2018, landfall="2018-09-14 11:15", place="Wrightsville Beach, NC",  cat="Cat 1", states=["37", "45"],                         days=21, prior="flood", atcf="AL062018"),
    dict(key="michael2018",  name="Michael",  year=2018, landfall="2018-10-10 17:30", place="Tyndall AFB, FL",         cat="Cat 5", states=["12", "13", "01", "45", "37"],       days=30, prior="wind",  atcf="AL142018"),
    dict(key="laura2020",    name="Laura",    year=2020, landfall="2020-08-27 06:00", place="Cameron, LA",             cat="Cat 4", states=["22", "48"],                         days=30, prior="wind",  atcf="AL132020"),
    dict(key="sally2020",    name="Sally",    year=2020, landfall="2020-09-16 09:45", place="Gulf Shores, AL",         cat="Cat 2", states=["01", "12", "13"],                   days=21, prior="mixed", atcf="AL192020"),
    dict(key="zeta2020",     name="Zeta",     year=2020, landfall="2020-10-28 21:00", place="Cocodrie, LA",            cat="Cat 3", states=["22", "28", "01", "13", "45", "37"], days=21, prior="wind",  atcf="AL282020"),
    dict(key="ida2021",      name="Ida",      year=2021, landfall="2021-08-29 16:55", place="Port Fourchon, LA",       cat="Cat 4", states=["22", "28"],                         days=30, prior="wind",  atcf="AL092021"),
    dict(key="ian2022",      name="Ian",      year=2022, landfall="2022-09-28 19:05", place="Cayo Costa, FL",          cat="Cat 4", states=["12", "45", "37"],                   days=21, prior="mixed", atcf="AL092022"),
    dict(key="beryl2024",    name="Beryl",    year=2024, landfall="2024-07-08 08:40", place="Matagorda, TX",           cat="Cat 1", states=["48"],                               days=21, prior="wind",  atcf="AL022024"),
    dict(key="helene2024",   name="Helene",   year=2024, landfall="2024-09-27 03:10", place="Perry, FL",               cat="Cat 4", states=["12", "13", "45", "37"],             days=30, prior="mixed", atcf="AL092024"),
]).set_index("key")
EVENTS["landfall"] = pd.to_datetime(EVENTS["landfall"])
EVENTS["report"] = "https://www.nhc.noaa.gov/data/tcr/" + EVENTS["atcf"] + "_" + EVENTS["name"] + ".pdf"

STATE_ABBR   = {"01": "AL", "12": "FL", "13": "GA", "22": "LA", "28": "MS", "37": "NC", "45": "SC", "48": "TX"}
STATE_CUTOFF = {"helene2024": {"12": "2024-10-09"}}        # Florida: up to the arrival of Milton

def event_window(key):
    # event window: PRE_DAYS days before landfall to `days` days after
    day0 = EVENTS.loc[key, "landfall"].normalize()
    return day0 - pd.Timedelta(days=PRE_DAYS), day0 + pd.Timedelta(days=int(EVENTS.loc[key, "days"]) + 1)

EVENTS[["name", "year", "landfall", "place", "cat", "states", "days", "prior", "report"]]

## B. Read the EAGLE-I outage records

The records each event needs are kept as `eaglei_<event>.csv` in the output folder. This cell reads those files; if one is missing, the EAGLE-I file of the year concerned is downloaded and filtered.

At the end, the cell also reads the header of each year file (only the first few hundred bytes) to confirm that the column names are the same in every year. If that fails, the later steps are not affected.

In [ ]:
# 4. Download EAGLE-I year by year and filter the records each event needs
EAGLEI_FILES = {2017: 42547828, 2018: 42547879, 2020: 42547894, 2021: 42547891, 2022: 42547897, 2024: 53581661}   # figshare file ids

def download(url, path, label="", min_bytes=1000):
    # download a file to path; skip it if it already exists
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return path
    print(f"Downloading {label or url} …")
    with requests.get(url, stream=True, timeout=180, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        done, shown = 0, 0
        with open(path + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block); done += len(block)
                if done - shown >= 400e6:
                    shown = done; print(f"  {done / 1e6:,.0f} MB downloaded")
    if done < min_bytes:
        os.remove(path + ".part")
        raise RuntimeError(f"The downloaded file is too small ({done} bytes); the site may have blocked the request. Open {url} in a browser, download it manually, "
                           f"place it in the us_flood_outage folder under the same name, then rerun this cell.")
    os.replace(path + ".part", path)
    return path

def parse_time(s):
    # the two timestamp styles that may appear in the files
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%y %H:%M", None):
        t = pd.to_datetime(s, format=fmt, errors="coerce")
        if t.notna().mean() > 0.99: break
    return t

def cache_path(key):
    return f"{OUT}/eaglei_{key}.csv"

for year, file_id in EAGLEI_FILES.items():
    todo = [k for k in EVENTS.index[EVENTS["year"] == year] if not os.path.exists(cache_path(k))]
    if not todo:
        continue
    raw = f"{OUT}/eaglei_outages_{year}.csv"                 # a manually downloaded copy can be placed here
    if not os.path.exists(raw):
        raw = download(f"https://ndownloader.figshare.com/files/{file_id}", f"{TMP}/eaglei_outages_{year}.csv",
                       f"EAGLE-I {year}", 300e6)
    header = {c.strip().lower(): c for c in pd.read_csv(raw, nrows=0, encoding="utf-8-sig").columns}
    fips_col, time_col = header["fips_code"], header["run_start_time"]
    out_col = header.get("customers_out", header.get("sum"))
    if out_col is None:
        raise KeyError(f"No customers-out column in the {year} file; the header is: {list(header.values())}")
    states = sorted({s for k in todo for s in EVENTS.loc[k, "states"]})
    parts = {k: [] for k in todo}
    for chunk in pd.read_csv(raw, usecols=[fips_col, out_col, time_col], dtype={fips_col: str},
                             encoding="utf-8-sig", chunksize=2_000_000):
        fips = chunk[fips_col].str.split(".").str[0].str.zfill(5)
        keep = fips.str[:2].isin(states)
        if not keep.any():
            continue
        sub = pd.DataFrame({"fips": fips[keep], "customers_out": pd.to_numeric(chunk.loc[keep, out_col], errors="coerce"),
                            "time": parse_time(chunk.loc[keep, time_col])}).dropna()
        for k in todo:
            t0, t1 = event_window(k)
            m = sub["fips"].str[:2].isin(EVENTS.loc[k, "states"]) & (sub["time"] >= t0) & (sub["time"] < t1)
            if m.any():
                parts[k].append(sub[m])
    for k in todo:
        one = pd.concat(parts[k]) if parts[k] else pd.DataFrame(columns=["fips", "customers_out", "time"])
        one.to_csv(cache_path(k), index=False)
    print(f"{year}: extracted {', '.join(todo)}")

ea = {}
for k in EVENTS.index:
    ea[k] = pd.read_csv(cache_path(k), dtype={"fips": str})
    ea[k]["time"] = pd.to_datetime(ea[k]["time"])

# header of each year file (reads only the first 400 bytes)
for year, file_id in EAGLEI_FILES.items():
    try:
        with requests.get(f"https://ndownloader.figshare.com/files/{file_id}", stream=True, timeout=30,
                          headers={"User-Agent": "Mozilla/5.0", "Range": "bytes=0-399"}) as r:
            r.raise_for_status()
            head = r.raw.read(400).decode("utf-8", "replace").splitlines()
        print(year, "|", " | ".join(head[:2]))
    except Exception as err:
        print(year, "header not read:", str(err)[:80])
pd.DataFrame({k: {"rows": len(d), "counties_with_records": d["fips"].nunique(),
                  "first": d["time"].min(), "last": d["time"].max()} for k, d in ea.items()}).T

## C. Customers per county (the denominator)

**County customers = county population in the event's own year × statewide customers per person in 2022**

- **Population is taken for the event's own year**: the two Census county population files cover 2010–2019 and 2020–2024
- **Customers per person always come from 2022**: that is, "statewide customers ÷ state population" for the latest year in `coverage_history.csv`. Customers per person in a given state should be about the same from year to year, but in this file the Texas figures for 2018–2020 are about 30% above those for 2021–2022, so the 2022 figure is used for every event year. The first table below lists customers per person by state and year and shows this inconsistency directly

What this cell computes is the prior estimate `customers_est`. After the cleaning in part D, when a county's peak customers out exceed this estimate, the peak is used instead (`customers_used`).

`MCC.csv` (EAGLE-I's own county customer table) is still read in, for comparison only: the second table shows that it is clearly too small in North Carolina and Texas, while in the other 6 states its statewide total is within 2% of the estimate used here.

In [ ]:
# 5. Estimated customers per county (by event year), compared with MCC.csv
CENSUS = "https://www2.census.gov/programs-surveys/popest/datasets"
CENSUS_URLS = [[f"{CENSUS}/2010-2019/counties/totals/co-est2019-alldata.csv"],
               [f"{CENSUS}/2020-2024/counties/totals/co-est2024-alldata.csv",      # the second address is a fallback
                f"{CENSUS}/2020-2023/counties/totals/co-est2023-alldata.csv"]]
COVERAGE_URL = "https://ndownloader.figshare.com/files/42547714"
MCC_URL      = "https://ndownloader.figshare.com/files/42547708"

def first_available(urls):
    # try the addresses in turn and return the first file that works
    for url in urls:
        name = url.rsplit("/", 1)[1]
        try:
            return download(url, f"{OUT}/{name}", name)
        except Exception as err:
            print(f"{name} could not be downloaded: {err}")
    raise RuntimeError(f"Download failed. Open {urls[0]} in a browser, download it manually, place it in the us_flood_outage folder, then rerun this cell.")

pops, names = [], pd.Series(dtype=object)
for urls in CENSUS_URLS:
    c = pd.read_csv(first_available(urls), encoding="latin-1", dtype={"STATE": str, "COUNTY": str})
    c = c[c["SUMLEV"] == 50].copy()
    c.index = c["STATE"].str.zfill(2) + c["COUNTY"].str.zfill(3)
    years = {col: int(col[-4:]) for col in c.columns if col.startswith("POPESTIMATE") and col[-4:].isdigit()}
    pops.append(c[list(years)].rename(columns=years))
    names = names.combine_first(c["CTYNAME"])
pop = pd.concat(pops, axis=1)                                    # rows = counties, columns = years
pop = pop[pop.index.str[:2].isin(list(STATE_ABBR))].dropna()
county_name = (names.reindex(pop.index).str.replace(" County", "", regex=False).str.replace(" Parish", "", regex=False)
               + ", " + pop.index.str[:2].map(STATE_ABBR))
print(f"Population: {len(pop)} counties, years {pop.columns.min()}–{pop.columns.max()}")

def pop_year(year):
    # the year in the population table closest to this one
    return min(pop.columns, key=lambda y: abs(y - year))

cov = pd.read_csv(download(COVERAGE_URL, f"{OUT}/coverage_history.csv", "coverage_history.csv"), encoding="utf-8-sig")
cov.columns = [c.strip().lower() for c in cov.columns]
txt = cov["year"].astype(str)                                    # the year is written "1/1/18" or "2018"; both become 2018
yr = pd.to_numeric(txt.str.extract(r"((?:19|20)\d\d)")[0], errors="coerce")
yr = yr.fillna(2000 + pd.to_numeric(txt.str.extract(r"(\d{1,2})\s*$")[0], errors="coerce"))
cov["year"]  = yr.astype(int)
cov["state"] = cov["state"].astype(str).str.strip()
for col in ["total_customers", "min_pct_covered", "max_pct_covered"]:
    cov[col] = pd.to_numeric(cov[col].astype(str).str.replace("[,%]", "", regex=True), errors="coerce")
    if col.endswith("pct_covered") and cov[col].max() > 1.5:     # if written as percentages, convert to shares between 0 and 1
        cov[col] = cov[col] / 100
REF_YEAR = int(cov["year"].max())                                # the latest year in the file (2022)
cov[cov["state"].isin(list(STATE_ABBR.values()))].sort_values(["state", "year"]).to_csv(f"{OUT}/step2c_coverage_by_state.csv", index=False)

def customers_per_person(state, year=None):
    # statewide customers ÷ state population; the latest year unless a year is given
    year = REF_YEAR if year is None else year
    row = cov[(cov["state"] == STATE_ABBR[state]) & (cov["year"] == year)]
    if row.empty:
        raise ValueError(f"coverage_history.csv has no record for {STATE_ABBR[state]} in {year}; the header is {list(cov.columns)}")
    return row["total_customers"].iloc[0] / pop.loc[pop.index.str[:2] == state, pop_year(year)].sum()

den, coverage = {}, {}
for key, ev in EVENTS.iterrows():
    parts = [pop.loc[pop.index.str[:2] == s, pop_year(ev["year"])] * customers_per_person(s) for s in ev["states"]]
    den[key] = pd.concat(parts).round().sort_index()
    c = cov[cov["state"].isin([STATE_ABBR[s] for s in ev["states"]]) & (cov["year"] == ev["year"])]
    coverage[key] = (f"{c['max_pct_covered'].min():.0%}–{c['max_pct_covered'].max():.0%}" if len(c)     # range of each state's highest coverage in that year
                     else "not published")

# customers per person by state and year: about the same from year to year in a given state, and between 0.4 and 0.6
ratio = pd.DataFrame({int(y): {STATE_ABBR[s]: customers_per_person(s, int(y)) for s in STATE_ABBR} for y in sorted(cov["year"].unique())})
ratio.to_csv(f"{OUT}/step2c_customers_per_person.csv")
display(tidy(ratio, 3))

# Comparison: in each state, how many counties have an MCC value below half of the estimate (for 2022, the year MCC refers to)
mcc = pd.read_csv(download(MCC_URL, f"{OUT}/MCC.csv", "MCC.csv"), encoding="utf-8-sig")
mcc.columns = ["fips", "mcc_customers"]
mcc = mcc[pd.to_numeric(mcc["fips"], errors="coerce").notna()].copy()
mcc["fips"] = mcc["fips"].astype(str).str.split(".").str[0].str.zfill(5)
est22 = pd.concat([pop.loc[pop.index.str[:2] == s, pop_year(2022)] * customers_per_person(s, 2022) for s in STATE_ABBR])
cmp = pd.DataFrame({"state": est22.index.str[:2].map(STATE_ABBR), "estimate": est22,
                    "mcc": mcc.set_index("fips")["mcc_customers"].astype(float).reindex(est22.index)})
cmp["ratio"] = cmp["mcc"] / cmp["estimate"]
mcc_tab = cmp.groupby("state").agg(counties=("estimate", "size"), estimate_total=("estimate", "sum"), mcc_total=("mcc", "sum"),
                                   median_ratio=("ratio", "median"), below_half=("ratio", lambda r: int((r < 0.5).sum())))
mcc_tab.to_csv(f"{OUT}/step2c_mcc_comparison.csv")
tidy(mcc_tab, 2)

## D. Cleaning; build the 15-minute panels

Each event gets one table: rows are 15-minute time slots, columns are counties, values are customers out. Cleaning takes four steps:

1. **Whole-state gaps**: EAGLE-I does not record "0 customers out", so a county with no record usually means 0. But when a state has no record at all at some time, it is treated as a collection gap and all counties of that state are left empty for that slot
2. **Filling short gaps**: gaps of up to 12 hours are filled by linear interpolation between the values on either side; longer gaps stay empty and are counted in the quality indicators
3. **Spikes**: when a county's number suddenly jumps above 1.5 times the median of the surrounding hour and comes back in under 1 hour, it is treated as double counting and replaced by the median. A rise that lasts 1 hour or more is left alone, because it is more likely a real outage
4. **Sharp drops**: when a county's number suddenly falls below half of the median of the surrounding 3 hours and quickly comes back, a utility was usually not collected at that moment, and the value is replaced by the median. A drop that lasts 3 hours or more is left alone, because it is more likely a real restoration

Steps 3 and 4 correct a value only when the deviation reaches at least 2% of the county's customers; values right next to a long gap, where the surrounding data are incomplete, are not corrected.

Two more things are done after the cleaning:

- **Share of time with a record** (`record_share`): for each county, the share of time slots that have a record within the period searched for the peak (2 days before landfall to 7 days after). A county where it equals 0 was most likely not covered by EAGLE-I at the time
- **Adjusting the denominator** (`customers_used`): a county cannot have more customers out than it has customers. When the cleaned peak exceeds the estimate from part C, the peak is used as the county's customer total for that event

The first table below shows the cleaning for each event; the second shows for how many counties per event the denominator was raised, and by how much.

In [ ]:
# 6. Clean and build the panel for each event
def run_lengths(flag):
    # for each True, the length of the run of consecutive Trues it belongs to (0 where False)
    flag = pd.Series(np.asarray(flag, dtype=bool))
    return flag.groupby((~flag).cumsum()).transform("sum").where(flag, 0).values

def peak_zone(key, index):
    # the period searched for the peak: 2 days before landfall to PEAK_DAYS days after
    t = EVENTS.loc[key, "landfall"]
    return (index >= t - pd.Timedelta(days=2)) & (index <= t + pd.Timedelta(days=PEAK_DAYS))

def build_panel(key):
    ev = EVENTS.loc[key]
    start, end = event_window(key)
    grid = pd.date_range(start, end, freq="15min", inclusive="left")
    cols = den[key].index
    state_of = cols.str[:2]
    d = ea[key].assign(slot=ea[key]["time"].dt.floor("15min"))
    seen = (d.groupby(["slot", "fips"])["customers_out"].max().unstack("fips")
              .reindex(index=grid, columns=cols))                # empty where there is no record

    # (1) whole-state gaps: the state has no record at all in this slot
    gap = pd.DataFrame(False, index=grid, columns=cols)
    for s in ev["states"]:
        has = grid.isin(d.loc[d["fips"].str[:2] == s, "slot"])
        gap.loc[~has, state_of == s] = True
    cut = pd.DataFrame(False, index=grid, columns=cols)         # data after the arrival of another hurricane are not used
    for s, day in STATE_CUTOFF.get(key, {}).items():
        cut.loc[pd.Timestamp(day):, state_of == s] = True
    valid = ~(gap | cut)
    raw = seen.fillna(0.0).where(valid)                         # the panel before cleaning
    zone = peak_zone(key, grid)
    info = pd.DataFrame({"record_share": (seen.notna() & valid)[zone].sum() / valid[zone].sum().clip(lower=1),
                         "peak_before_cleaning": raw[zone].max()})
    wide, floor = raw.copy(), DROP_MIN_SHARE * den[key]

    # (2) fill short gaps: interpolate everything, then empty the long gaps again
    limit = int(GAP_FILL_H / STEP_H)
    long_gap = pd.DataFrame(False, index=grid, columns=cols)
    for s in ev["states"]:
        g = gap.loc[:, state_of == s].iloc[:, 0]
        long_gap.loc[run_lengths(g) > limit, state_of == s] = True
    wide = wide.interpolate(method="time", limit_area="inside").mask(long_gap | cut)

    # (3) spikes: values far above the surrounding median that last under 1 hour are replaced by the median (left alone where the surrounding data are incomplete)
    win = int(SPIKE_WINDOW_H / STEP_H) + 1
    med = wide.rolling(win, center=True, min_periods=win).median()
    spike = (wide > SPIKE_RATIO * med) & (wide - med).ge(floor, axis=1)
    wide = wide.mask(spike, med)

    # (4) sharp drops: values far below the surrounding median are replaced by the median
    win = int(DROP_WINDOW_H / STEP_H) + 1
    med = wide.rolling(win, center=True, min_periods=win).median()
    drop = (wide < DROP_RATIO * med) & (med - wide).ge(floor, axis=1)
    wide = wide.mask(drop, med)

    quality = {
        "filled_gap_h": float((gap & ~long_gap).any(axis=1).sum() * STEP_H),      # gaps that were filled
        "long_gap_h": float(long_gap.any(axis=1).sum() * STEP_H),                 # long gaps left empty (whole window)
        "long_gap_peak_h": float(long_gap[zone].any(axis=1).sum() * STEP_H),      # of which inside the period searched for the peak
        "spike_fixed_pct": float(spike.values.mean() * 100),                      # share of slots corrected as spikes
        "drop_fixed_pct": float(drop.values.mean() * 100),                        # share of slots corrected as sharp drops
        "peak_total_raw_k": float(raw.sum(axis=1, skipna=False).max() / 1000),    # peak of the total before cleaning (thousands)
    }
    return wide, raw, quality, info

panels, raw_panels, quality, county_info, den_used = {}, {}, {}, {}, {}
for key in EVENTS.index:
    panels[key], raw_panels[key], quality[key], county_info[key] = build_panel(key)
    # a county cannot have more customers out than it has customers: when the peak exceeds the estimate, the peak becomes the denominator
    peak = panels[key][peak_zone(key, panels[key].index)].max().fillna(0)
    den_used[key] = np.maximum(den[key], peak)
display(tidy(pd.DataFrame(quality).T, 3))

raised = pd.DataFrame({k: {"counties": len(den[k]), "denominator_raised": int((den_used[k] > den[k]).sum()),
                           "median_raise_pct": ((den_used[k] / den[k] - 1)[den_used[k] > den[k]] * 100).median(),
                           "max_raise_pct": (den_used[k] / den[k] - 1).max() * 100} for k in EVENTS.index}).T
tidy(raised)

## E. The "county × event" label table

Each row describes the outage in one county during one event. The main columns:

| Column | Meaning |
| --- | --- |
| `customers_est`, `customers_used` | The prior estimate of the customer total; the customer total actually used as the denominator (the peak, when it exceeds the estimate) |
| `peak_pct` | Peak outage share (%), searched from 2 days before landfall to 7 days after |
| `peak_before_cleaning` | Peak customers out before cleaning. Where it is far above `peak_customers_out`, the original peak was a spike |
| `hours_above` | Cumulative hours with an outage share ≥ 10%, counted up to restoration |
| `hours_if_customers_low`, `hours_if_customers_high` | What `hours_above` would be if the true customer total were 25% lower or 25% higher than the value used |
| `days_to_restore` | Days after landfall until restoration. Restoration is the first time after the peak that stays below 10% for 24 hours |
| `hours_rel` | Cumulative hours with customers out at or above 10% of the county's own peak. This indicator does not depend on the denominator and is computed for affected counties only |
| `affected` | Affected county: peak outage share ≥ 10% and at least 100 customers out at the peak |
| `censored`, `censored_rel` | Affected but not restored within the observation window (for `hours_above` and `hours_rel` respectively). Durations for these counties are underestimates |
| `denom_raised` | The denominator was raised, so the prior estimate was too small |
| `record_share`, `no_record` | Share of time with a record from 2 days before landfall to 7 days after; no record at all in that period |
| `state_ok` | The state has data for this event. A state where more than 20% of the counties have no record at all is marked `False` as a whole |
| `missing_hours` | Hours of empty data during the outage |

**Notes**

- The statistics, figures and screening that follow use only the rows where `state_ok` is `True`. The states removed are listed in this cell's output
- `affected` looks at the outage alone and does not judge whether this hurricane caused it. A county far from the hurricane that lost power for another reason at the same time is counted as well. Step 3 adds wind speed and rainfall for each county and screens such counties out
- A `no_record` county should not be treated as "no outage" in the modeling, even when its state has data

In [ ]:
# 7. Compute the outage labels for each county in each event
CALM_SLOTS = int(CALM_HOURS / STEP_H)

def first_calm(below, after_peak):
    # the first time after the peak from which the condition holds for the next CALM_HOURS hours; NaT if there is none
    calm = below.astype(float)[::-1].rolling(CALM_SLOTS, min_periods=CALM_SLOTS).min()[::-1].eq(1) & after_peak
    return calm.idxmax().where(calm.any())

def label_event(key, den_scale=1.0):
    # den_scale: multiply the denominator by this number, for the sensitivity check
    ev, wide, d = EVENTS.loc[key], panels[key], den_used[key] * den_scale
    idx = wide.index
    zone = peak_zone(key, idx)
    peak_out  = wide[zone].max()
    peak_time = wide[zone].fillna(0).idxmax()
    after_peak = pd.DataFrame(idx.values[:, None] >= peak_time.values[None, :], index=idx, columns=wide.columns)
    end = idx[-1] + pd.Timedelta(minutes=15)

    def spell(level, thresh):
        # for one "outage level": which slots are at or above the threshold, when it is restored, and the slots before restoration
        restored = first_calm(level < thresh, after_peak)
        until = pd.DataFrame(idx.values[:, None] < restored.fillna(end).values[None, :], index=idx, columns=wide.columns)
        return level >= thresh, restored, until

    share = wide.div(d, axis=1).clip(upper=1.0)                                 # share of customers
    rel   = wide.div(peak_out.where(peak_out > 0), axis=1)                      # share of the county's own peak
    above, restored, until = spell(share, OUT_THRESH)
    above_rel, restored_rel, until_rel = spell(rel, REL_THRESH)
    onset = above.idxmax().where(above.any())
    since = pd.DataFrame(idx.values[:, None] >= onset.fillna(end).values[None, :], index=idx, columns=wide.columns)

    out = pd.DataFrame({
        "event": key, "fips": wide.columns, "name": county_name.reindex(wide.columns).values,
        "state": wide.columns.str[:2].map(STATE_ABBR), "year": ev["year"],
        "customers_est": den[key].values, "customers_used": d.values,
        "peak_customers_out": peak_out.values, "peak_before_cleaning": county_info[key]["peak_before_cleaning"].values,
        "peak_pct": share[zone].max().values * 100,
        "peak_time": peak_time.values, "onset_time": onset.values, "restored_time": restored.values,
        "hours_above": (above & until).sum().values * STEP_H,
        "days_to_restore": ((restored - ev["landfall"]) / pd.Timedelta(days=1)).values,
        "hours_rel": (above_rel & until_rel).sum().values * STEP_H,
        "customer_hours": wide.where(until).sum().values * STEP_H,
        "missing_hours": (wide.isna() & since & until).sum().values * STEP_H,
        "record_share": county_info[key]["record_share"].values,
    })
    out["affected"] = (out["peak_pct"] >= OUT_THRESH * 100) & (out["peak_customers_out"] >= MIN_PEAK)
    out["censored"] = out["affected"] & out["restored_time"].isna()
    out["censored_rel"] = out["affected"] & restored_rel.isna().values
    out["hours_rel"] = out["hours_rel"].where(out["affected"])                  # meaningful for affected counties only
    out["denom_raised"] = out["customers_used"] > out["customers_est"]
    out["no_record"] = out["record_share"] == 0
    return out

labels = pd.concat([label_event(k) for k in EVENTS.index], ignore_index=True)
for col, scale in [("hours_if_customers_low", 1 - DEN_ERR), ("hours_if_customers_high", 1 + DEN_ERR)]:
    labels[col] = pd.concat([label_event(k, scale) for k in EVENTS.index], ignore_index=True)["hours_above"]

# which states have no data in which event: the share of counties without any record exceeds NO_RECORD_MAX
state_tab = labels.groupby(["event", "state"], sort=False).agg(
    counties=("fips", "size"), no_record=("no_record", "sum"), affected=("affected", "sum")).reset_index()
state_tab["no_record_pct"] = state_tab["no_record"] / state_tab["counties"] * 100
state_tab["state_ok"] = state_tab["no_record_pct"] <= NO_RECORD_MAX * 100
state_tab.to_csv(f"{OUT}/step2c_records_by_state.csv", index=False)
labels = labels.merge(state_tab[["event", "state", "state_ok"]], on=["event", "state"], how="left")
labels.to_csv(f"{OUT}/step2c_labels_county_event.csv", index=False)

dropped = state_tab[~state_tab["state_ok"]]
print("States removed for lack of data:",
      ", ".join(f"{r.event} {r.state} ({r.no_record_pct:.0f}%)" for r in dropped.itertuples()) or "none")
aff = labels[labels["affected"] & labels["state_ok"]]
print(f"Label table: {len(labels):,} rows (county × event); {len(aff):,} affected rows in states with data, {int((aff['hours_above'] >= 24).sum()):,} of them with outages over 24 hours, {int((aff['hours_above'] >= 168).sum()):,} over 7 days")
display(tidy(state_tab[state_tab["no_record"] > 0].sort_values("no_record_pct", ascending=False).head(12)))
tidy(aff.sort_values("hours_above", ascending=False).head(15)[
    ["event", "name", "customers_est", "customers_used", "peak_pct", "hours_above", "hours_rel", "censored"]])

In [ ]:
# 8. Figure 1: outage curve of each event (aligned by days since landfall)
keys = list(EVENTS.index)
ncol = 4
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 2.5 * nrow))
for ax, key in zip(axes.flat, keys):
    ev = EVENTS.loc[key]
    total = panels[key].sum(axis=1, skipna=False) / 1000                        # thousands of customers; left blank when any state has no data
    x = (total.index - ev["landfall"]) / pd.Timedelta(days=1)
    ax.axvline(0, color=FAINT, lw=1)
    ax.plot(x, total.values, color=C_POWER, lw=1.8)
    ax.set_title(f"{ev['name']} {ev['year']}")
    ax.text(0.97, 0.93, f"peak {total.max():,.0f}k\n" + ", ".join(STATE_ABBR[s] for s in ev["states"]),
            transform=ax.transAxes, ha="right", va="top", color=MUTED, fontsize=8.5, linespacing=1.4)
    ax.set_xlim(-PRE_DAYS, 31); ax.set_ylim(bottom=0); ax.set_xticks([0, 7, 14, 21, 28])
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)
for ax in axes.flat[max(len(keys) - ncol, 0):len(keys)]:
    ax.set_xlabel("Days since landfall")
fig.suptitle("Customers without power by event (thousands)", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step2c_fig1_event_curves.png"); plt.show()

# save the total curves (one value per hour) for comparison with official figures: for whole events and by state
def hourly_max(series, **tags):
    return series.resample("1h").max().rename("customers_out").reset_index().rename(columns={"index": "time"}).assign(**tags)

totals = pd.concat([hourly_max(panels[k].sum(axis=1, skipna=False), event=k) for k in keys])
totals[["event", "time", "customers_out"]].to_csv(f"{OUT}/step2c_event_totals.csv", index=False)
by_state = pd.concat([hourly_max(panels[k].loc[:, panels[k].columns.str[:2] == s].sum(axis=1, skipna=False), event=k, state=STATE_ABBR[s])
                      for k in keys for s in EVENTS.loc[k, "states"]])
by_state[["event", "state", "time", "customers_out"]].to_csv(f"{OUT}/step2c_state_totals.csv", index=False)

# hourly curves of the affected counties: the mean after cleaning, and the maximum before cleaning (to check what the cleaning changed)
parts = []
for k in keys:
    cols = aff.loc[aff["event"] == k, "fips"].tolist()
    if cols:
        clean = panels[k][cols].resample("1h").mean().round(1).reset_index().melt(id_vars="index", var_name="fips", value_name="customers_out")
        before = raw_panels[k][cols].resample("1h").max().reset_index().melt(id_vars="index", var_name="fips", value_name="raw_max")
        parts.append(clean.assign(raw_max=before["raw_max"].values, event=k).rename(columns={"index": "time"}))
hourly = pd.concat(parts)[["event", "fips", "time", "customers_out", "raw_max"]]
hourly.to_csv(f"{OUT}/step2c_panel_hourly.csv.gz", index=False, compression="gzip")
print(f"Hourly curves saved: {len(hourly):,} rows")

In [ ]:
# 9. Figure 2: distribution of outage duration among affected counties, by event
order = list(EVENTS.index)[::-1]                                                # earliest event at the top
xmax = float(aff["hours_above"].max())
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(11, 0.52 * len(order) + 1.3))
for v, lab in [(24, "24 h"), (168, "7 days")]:
    ax.axvline(v, color=FAINT, lw=1)
    ax.text(v, len(order) - 0.35, " " + lab, color=MUTED, fontsize=9, va="bottom")
for i, key in enumerate(order):
    a = aff[aff["event"] == key]
    h, y, done = a["hours_above"].values, i + rng.uniform(-0.2, 0.2, len(a)), ~a["censored"].values
    ax.plot(h[done], y[done], ls="none", marker="o", ms=5, mfc=C_POWER, mec=SURFACE, mew=0.7, alpha=0.8)
    ax.plot(h[~done], y[~done], ls="none", marker="o", ms=5, mfc=SURFACE, mec=C_POWER, mew=1.2)     # hollow = not restored within the window
    ax.text(1.02, i, f"{len(a)} counties, {int((h >= 168).sum())} at 7+ days", transform=ax.get_yaxis_transform(),
            va="center", color=INK, fontsize=9)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([f"{EVENTS.loc[k, 'name']} {EVENTS.loc[k, 'year']}" for k in order], color=INK)
ax.set_ylim(-0.6, len(order) + 0.1); ax.set_xlim(0, xmax * 1.03); ax.grid(axis="y", visible=False)
ax.set_xlabel(f"Hours with at least {OUT_THRESH:.0%} of customers out\n(one dot per affected county; hollow = power not restored within the window)")
ax.set_title("Outage duration by event")
fig.savefig(f"{OUT}/step2c_fig2_duration_by_event.png"); plt.show()

In [ ]:
# 10. Figure 3: how sensitive the duration is to the denominator; comparison with the denominator-free hours_rel
rho_low  = spearman(aff["hours_above"], aff["hours_if_customers_low"])
rho_high = spearman(aff["hours_above"], aff["hours_if_customers_high"])
rho_den  = min(rho_low, rho_high)
both     = aff[~aff["censored"] & ~aff["censored_rel"]]
rho_rel  = spearman(both["hours_above"], both["hours_rel"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5.6))
lim = float(aff[["hours_above", "hours_if_customers_low"]].max().max()) * 1.05
many = len(aff) > 400                                                           # with many counties, draw lines and dots lighter
ax1.plot([0, lim], [0, lim], color=FAINT, lw=1)
ax1.vlines(aff["hours_above"], aff["hours_if_customers_high"], aff["hours_if_customers_low"], color=C_POWER,
           lw=0.8 if many else 1.1, alpha=0.2 if many else 0.45)
ax1.set_xlim(0, lim); ax1.set_ylim(0, lim); ax1.set_aspect("equal")
ax1.set_xlabel(f"Hours with at least {OUT_THRESH:.0%} out, customer totals as used")
ax1.set_ylabel(f"Same, with totals {DEN_ERR:.0%} higher (bottom of line) to {DEN_ERR:.0%} lower (top)")
ax1.set_title(f"If customer totals were {DEN_ERR:.0%} off", pad=22)
ax1.text(0, 1.03, f"One line per affected county (n = {len(aff)}), Spearman {rho_den:.2f} or higher",
         transform=ax1.transAxes, color=MUTED, fontsize=9)

lim = float(max(both["hours_above"].max(), both["hours_rel"].max())) * 1.05
ax2.plot([0, lim], [0, lim], color=FAINT, lw=1)
ax2.scatter(both["hours_above"], both["hours_rel"], s=10 if many else 22, color=C_POWER, edgecolor=SURFACE,
            linewidth=0.4 if many else 0.6, alpha=0.6 if many else 0.75)
ax2.set_xlim(0, lim); ax2.set_ylim(0, lim); ax2.set_aspect("equal")
ax2.set_xlabel(f"Hours with at least {OUT_THRESH:.0%} of customers out")
ax2.set_ylabel(f"Hours with at least {REL_THRESH:.0%} of the county's own peak out")
ax2.set_title("Against a measure that needs no customer totals", pad=22)
ax2.text(0, 1.03, f"Affected counties with restoration observed (n = {len(both)}), Spearman {rho_rel:.2f}",
         transform=ax2.transAxes, color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(f"{OUT}/step2c_fig3_denominator_check.png"); plt.show()

by_event = pd.Series({k: spearman(g["hours_above"], g["hours_rel"]) for k, g in both.groupby("event") if len(g) >= 5})
print(f"Spearman correlation of the duration ranking after lowering / raising the denominator by {DEN_ERR:.0%}: {rho_low:.2f} / {rho_high:.2f}")
print(f"Spearman correlation between hours_above and hours_rel: all events {rho_rel:.2f}; by event:")
print(by_event.reindex(list(EVENTS.index)).round(2).to_string())

**Notes on the three figures**

- **Figure 1**: curves rise steeply around landfall and then fall back gradually. A break in a curve marks a state without data: a long gap that was not filled, or the period after Helene's cut-off date in Florida
- **Figure 2**: each dot is one affected county; only states with data are drawn. A wide spread means that outage duration varies a lot across the counties of that event. Hollow dots are counties not restored within the observation window; their true duration is longer than shown. The numbers on the right are the count of affected counties and of counties with outages of 7 days or more
- **Figure 3, left**: each vertical line is one county. A short line close to the diagonal means that a 25% error in the denominator barely changes that county's outage duration. A correlation above 0.9 means the ranking of counties hardly depends on the denominator
- **Figure 3, right**: the threshold of `hours_rel` is 10% of the county's own peak, which is never higher than the threshold of `hours_above`, so the dots lie on or above the diagonal. For counties whose peak is close to 100% the two are almost equal; the farther a dot is from the diagonal, the lower that county's peak outage share, or the more the data cover only part of its customers

## F. Sentinel-1 overpass check

This part first reads any saved overpass results (`step2c_s1_by_county.csv`), then checks which affected counties have no result yet, and connects to Earth Engine only for those.

- No county missing: no connection to Earth Engine
- Counties missing but no connection (for example `PROJECT_ID` not filled in): their result stays empty and the later steps run as usual

The definitions are unchanged: a **usable** image means the same orbit also has an image between 30 and 2 days before landfall; a county has a **timely** image when its first usable image comes no later than day 6 after landfall.

In [ ]:
# 11. [EE] Read the existing Sentinel-1 results; sign in to Earth Engine only if counties are missing
S1_COLS  = ["event", "fips", "s1_first_day", "s1_usable_images", "s1_orbits", "s1_any_first_day"]
S1_FILES = [f"{OUT}/step2c_s1_by_county.csv"]
found = [] if RERUN_S1 else [pd.read_csv(f, dtype={"fips": str})[S1_COLS] for f in S1_FILES if os.path.exists(f)]
s1_have = pd.concat(found).drop_duplicates(["event", "fips"]) if found else pd.DataFrame(columns=S1_COLS)
s1_have = s1_have.merge(aff[["event", "fips"]], on=["event", "fips"])          # keep only the counties that are affected under the current label definitions
todo = aff[["event", "fips"]].merge(s1_have[["event", "fips"]], on=["event", "fips"], how="left", indicator=True)
todo = todo.loc[todo["_merge"] == "left_only", ["event", "fips"]]
print(f"{len(aff):,} affected rows, {len(s1_have):,} with an existing Sentinel-1 result, {len(todo):,} still missing")

EE_OK = False
if len(todo):
    try:
        import ee
        ee.Authenticate()                 # follow the link to authorize
        ee.Initialize(project=PROJECT_ID)
        EE_OK = True
        print("Earth Engine initialized")
    except Exception as err:
        print("Could not connect to Earth Engine; the missing counties stay empty for now. Check PROJECT_ID in cell 2. Reason:", str(err)[:200])

In [ ]:
# 12. [EE] Compute the Sentinel-1 overpasses for the missing counties
def s1_passes(fips, start, end):
    # for each county: time and orbit number of every image in the period that covers the county's center point
    col = s1_all.filterDate(start, end)
    def one(f):
        hits = col.filterBounds(f.geometry().centroid(100))
        return ee.Feature(None, {"GEOID": f.get("GEOID"),
                                 "t": hits.aggregate_array("system:time_start"),
                                 "orbit": hits.aggregate_array("relativeOrbitNumber_start")})
    rows = []
    for i in range(0, len(fips), 150):                           # request in batches of 150 counties
        feats = ee_counties.filter(ee.Filter.inList("GEOID", fips[i:i + 150])).map(one).getInfo()["features"]
        for x in feats:
            pr = x["properties"]
            rows += [(pr["GEOID"], t, o) for t, o in zip(pr.get("t", []), pr.get("orbit", []))]
    return pd.DataFrame(rows, columns=["fips", "t", "orbit"])

s1_rows = []
if EE_OK:
    ee_counties = ee.FeatureCollection("TIGER/2018/Counties")
    REGION = ee.Geometry.Rectangle([-107, 24, -75, 37.5])        # a rectangle that covers the 8 states
    s1_all = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(REGION)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV")))
    for key, ev in EVENTS.iterrows():
        fips = todo.loc[todo["event"] == key, "fips"].tolist()
        if not fips:
            continue
        day0 = ev["landfall"].normalize()
        p = s1_passes(fips, (day0 - pd.Timedelta(days=30)).strftime("%Y-%m-%d"), (day0 + pd.Timedelta(days=16)).strftime("%Y-%m-%d"))
        p["days"] = (pd.to_datetime(p["t"], unit="ms") - ev["landfall"]) / pd.Timedelta(days=1)
        by_county = dict(tuple(p.groupby("fips")))
        for f in fips:
            q = by_county.get(f, p.iloc[:0])
            before = set(q.loc[q["days"] <= -2, "orbit"])                       # orbits with an image before landfall
            after = q[q["days"] >= 0]
            usable = after[after["orbit"].isin(before)]                         # usable = the same orbit was also imaged beforehand
            s1_rows.append({"event": key, "fips": f,
                            "s1_first_day": usable["days"].min(),               # days after landfall of the first usable image (empty if none)
                            "s1_usable_images": len(usable), "s1_orbits": usable["orbit"].nunique(),
                            "s1_any_first_day": after["days"].min()})           # first image when no earlier image is required
        print(f"{key}: computed {len(fips)} counties")

parts = [d for d in (s1_have, pd.DataFrame(s1_rows, columns=S1_COLS)) if len(d)]
s1_county = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=S1_COLS)
s1_county.to_csv(S1_FILES[0], index=False)
print(f"{len(aff):,} affected rows, {len(s1_county):,} of them with a Sentinel-1 result")
tidy(s1_county.groupby("event")[["s1_first_day", "s1_any_first_day", "s1_usable_images"]].median().reindex(list(EVENTS.index)))

## G. Event screening table and checklist

Each event is ticked against four criteria. The `coverage` column is the range of the highest coverage in that year across the event's states. All counts use only the states with data (`state_ok`).

| Criterion | Meaning | Threshold |
| --- | --- | --- |
| `ok_labels` | Enough labels: counties with outages over 24 hours | ≥ 20 |
| `ok_quality` | No big holes in the data: unfilled gaps from 2 days before landfall to 7 days after | 0 hours |
| `ok_ntl` | Night-light comparison possible: counties with outages over 7 days | ≥ 5 |
| `ok_s1` | Timely radar images: share of affected counties with a usable Sentinel-1 image within 6 days of landfall | ≥ 50% |

`usable` means the event can go into modeling, and requires both `ok_labels` and `ok_quality`. `ok_ntl` and `ok_s1` decide whether the event can also serve the night-light comparison and the flood feature.

Columns in the table: `states_dropped` (states removed for lack of data), `no_record` (counties that still have no record at all within the states with data), `denom_raised` (affected counties whose denominator was raised), `peak_total_raw_k` (peak of the total before cleaning; a large difference from `peak_total_k` means spikes).

In [ ]:
# 13. Event screening table
rows = []
for key, ev in EVENTS.iterrows():
    L = labels[(labels["event"] == key) & labels["state_ok"]]
    A = L[L["affected"]]
    gone = state_tab[(state_tab["event"] == key) & ~state_tab["state_ok"]]["state"]
    rows.append({
        "event": key, "prior": ev["prior"], "states_dropped": ", ".join(gone) or "—",
        "counties": len(L), "no_record": int(L["no_record"].sum()), "affected": len(A),
        "over_24h": int((A["hours_above"] >= 24).sum()), "over_7d": int((A["hours_above"] >= 168).sum()),
        "max_hours": A["hours_above"].max(), "median_hours_rel": A["hours_rel"].median(),
        "censored": int(A["censored"].sum()), "denom_raised": int(A["denom_raised"].sum()),
        "peak_total_k": panels[key].sum(axis=1, skipna=False).max() / 1000,
        "coverage": coverage[key], **quality[key],
    })
screen = pd.DataFrame(rows).set_index("event")

s1_tab = s1_county.groupby("event").agg(
    s1_counties=("fips", "size"),
    s1_share_pct=("s1_first_day", lambda d: (d <= S1_DAYS).mean() * 100),       # share of affected counties with a timely usable image
    s1_median_first_day=("s1_first_day", "median"),
    s1_no_image=("s1_first_day", lambda d: int(d.isna().sum())))                # counties without any usable image within 16 days
screen = screen.join(s1_tab.reindex(columns=["s1_counties", "s1_share_pct", "s1_median_first_day", "s1_no_image"]))
# add each county's Sentinel-1 status to the label table for Step 3
labels = (labels.drop(columns=["s1_first_day", "s1_orbits"], errors="ignore")
                .merge(s1_county[["event", "fips", "s1_first_day", "s1_orbits"]], on=["event", "fips"], how="left"))
labels.to_csv(f"{OUT}/step2c_labels_county_event.csv", index=False)

screen["ok_labels"]  = screen["over_24h"] >= 20
screen["ok_quality"] = screen["long_gap_peak_h"] == 0
screen["ok_ntl"]     = screen["over_7d"] >= 5
screen["ok_s1"]      = pd.to_numeric(screen["s1_share_pct"], errors="coerce") >= S1_MIN_SHARE * 100     # False when there is no result
screen["usable"]     = screen["ok_labels"] & screen["ok_quality"]
screen.to_csv(f"{OUT}/step2c_event_screening.csv")
tidy(screen)

In [ ]:
# 14. Step 2c checklist
def verdict(passed):
    return "pass" if passed else "fail"

use = screen[screen["usable"]]
n_use   = len(use)
n_rows  = int(use["affected"].sum())
n_ntl   = int(use["ok_ntl"].sum())
n_water = int(use["prior"].isin(["flood", "mixed"]).sum())
n_wind  = int((use["prior"] == "wind").sum())
n_s1    = int(use["ok_s1"].sum())

checks = [
    ("Usable events", n_use, "≥ 6", n_use >= 6),
    ("Affected county × event rows in usable events", n_rows, "≥ 500", n_rows >= 500),
    ("Usable events that allow a night-light comparison", n_ntl, "≥ 3", n_ntl >= 3),
    ("Usable events of flood or mixed type / of wind type", f"{n_water} / {n_wind}", "≥ 2 / ≥ 2", n_water >= 2 and n_wind >= 2),
    ("Usable events with timely Sentinel-1 images", n_s1, "≥ 4", n_s1 >= 4),
    ("Spearman correlation of the duration ranking when the denominator is 25% off", round(float(rho_den), 2), "≥ 0.9", rho_den >= 0.9),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step2c_checklist.csv", index=False)
print("Usable events:", ", ".join(use.index))
check_tab

## Output files

All written to the `us_flood_outage` folder:

- `step2c_labels_county_event.csv`, `step2c_event_screening.csv`, `step2c_checklist.csv`, `step2c_records_by_state.csv`
- `step2c_event_totals.csv`, `step2c_state_totals.csv`, `step2c_panel_hourly.csv.gz`
- `step2c_coverage_by_state.csv`, `step2c_mcc_comparison.csv`, `step2c_customers_per_person.csv`
- `step2c_fig1_event_curves.png`, `step2c_fig2_duration_by_event.png`, `step2c_fig3_denominator_check.png`